# PCA (Principal components analysis)

### What is it? (The Flashlight Analogy)
Imagine you have a 3D bicycle, a flashlight, and a blank wall. Your goal is to cast a shadow of the bicycle onto the wall that makes it as easy to recognize as possible.

* If you point the flashlight directly at the **front** of the bike, the shadow on the wall just looks like a skinny `T`-shaped line. You've lost almost all the information about the object.
* If you walk around and shine the flashlight at the **side** of the bike, the shadow clearly shows two wheels, the frame, and the handlebars. 

In both cases, you compressed a 3D object into a 2D shadow. But in the second case, you found the optimal angle that preserved the **most information** (or in data terms, the most *variance*).

PCA is an algorithm that mathematically finds that perfect "flashlight angle" for massive datasets. It allows us to squash data down from hundreds or thousands of dimensions into just a few, while deliberately keeping the most important patterns intact.

### The Formal Mathematical Definition
Formally, PCA is an orthogonal linear transformation that projects data into a new coordinate system. It ensures that the greatest variance by any scalar projection of the data lies on the first coordinate (the first principal component), the second greatest variance on the second coordinate, and so on.

Let our data be a mean-centered matrix $X$ of shape $n \times d$ (where $n$ is the number of samples and $d$ is the number of features). We want to find a unit weight vector $w$ (where $||w|| = 1$) that maximizes the variance of the projected data $Xw$. The variance of this projection is governed by the sample covariance matrix $\Sigma$:

$$\Sigma = \frac{1}{n-1} X^T X$$

The first principal component $w_1$ is found by solving the following optimization problem:

$$w_1 = \arg\max_{||w||=1} w^T \Sigma w$$

Using Lagrange multipliers, it can be proven that the solution to this problem is exactly the **dominant eigenvector** of the covariance matrix $\Sigma$, and the maximum variance achieved is its corresponding **eigenvalue**, $\lambda$. Successive principal components are simply the remaining eigenvectors, sorted by their eigenvalues in descending order.

### Why do we use it?

* **Dimensionality Reduction:** The most fundamental use. PCA mathematically projects a high-dimensional dataset onto a lower-dimensional subspace while preserving as much data variance (information) as possible.
* **Data Visualization:** Humans are visually limited to 3 dimensions. If you have a dataset with 50 features, you cannot simply graph it. PCA allows us to project those 50 dimensions down to 2 or 3 principal components, letting us plot the data to visually inspect for clusters, natural separations, or outliers.
* **Computational Efficiency:** The "Curse of Dimensionality" dictates that as the number of features increases, distance-based algorithms (like K-Means or K-Nearest Neighbors) become exponentially slower and require massively more data. By shrinking the feature space, PCA drastically speeds up training times and reduces memory footprint.
* **Eliminating Multicollinearity:** The math behind PCA guarantees that every principal component is **orthogonal** (perpendicular) to the others. This means the new features are perfectly uncorrelated ($Covariance = 0$). This is highly beneficial for algorithms that struggle with highly correlated inputs, such as Multiple Linear Regression or Naive Bayes.
* **Noise Filtering (Regularization):** In most datasets, the axes with the highest variance contain the actual "signal" or structure, while the axes with the lowest variance represent random background noise. By dropping the smallest principal components, PCA inherently acts as a low-pass data filter, helping downstream models generalize better and avoid overfitting.

<details>
<summary><b><font size="+1">The Proof: Why Eigenvectors?</font></b></summary>

We stated above that finding the axis of maximum variance magically reduces to finding the eigenvectors of the covariance matrix. Let's prove it mathematically using **Lagrange Multipliers**.

### 1. The Setup
Let $X$ be our mean-centered data matrix. We want to find a direction (a vector $w$) to project our data onto. The variance of our projected data is given by:
$$V = w^T \Sigma w$$
where $\Sigma$ is the covariance matrix ($\Sigma = \frac{1}{n-1}X^T X$).

**Our Goal:** Maximize $w^T \Sigma w$.

### 2. The Constraint
If we just try to maximize $w^T \Sigma w$ without rules, the math will just make $w$ infinitely large. We only care about the *direction*, not the length, so we must constrain $w$ to be a unit vector (length of 1). 
Mathematically, this constraint is written as:
$$w^T w = 1$$

### 3. The Lagrangian
To maximize a function subject to an equality constraint, we use a Lagrangian. We introduce a Lagrange multiplier, $\lambda$, and subtract the constraint from our objective function:
$$\mathcal{L}(w, \lambda) = w^T \Sigma w - \lambda(w^T w - 1)$$

### 4. The Derivative
To find the maximum, we take the partial derivative of the Lagrangian with respect to our vector $w$ and set it to zero.

*Note on matrix calculus: The derivative of $x^T A x$ with respect to $x$ is $2Ax$ (if $A$ is symmetric, which covariance matrices always are). The derivative of $x^T x$ is $2x$.*

$$\frac{\partial \mathcal{L}}{\partial w} = 2\Sigma w - 2\lambda w = 0$$

Divide by 2 and rearrange the equation:
$$\Sigma w = \lambda w$$

### 5. The Grand Conclusion
Look closely at the final equation: **$\Sigma w = \lambda w$**

This is the fundamental definition of an **eigenvector** and an **eigenvalue**! 
* $w$ is the eigenvector of the covariance matrix $\Sigma$.
* $\lambda$ is the corresponding eigenvalue.

Furthermore, if we want to know what the actual maximum variance *is*, we can plug $\Sigma w = \lambda w$ back into our original variance equation:
$$V = w^T (\Sigma w) = w^T (\lambda w) = \lambda (w^T w)$$
Since we constrained $w^T w = 1$, the equation simplifies to:
$$V = \lambda$$

**The Proof is Complete:** The variance of the projected data is exactly equal to the eigenvalue $\lambda$. Therefore, to maximize variance, we simply calculate the eigenvectors of the covariance matrix and choose the one with the largest eigenvalue. That is our **First Principal Component**.

</details>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy.testing import assert_almost_equal
from sklearn.datasets import load_sample_image
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42) # Set a random seed for reproducibility
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

### The Geometric Truth: PCA vs. Least Squares (OLS)

A very common misconception is that the first Principal Component is the same as a line of best fit (Linear Regression). Let's clear that up visually by looking at exactly what each algorithm is trying to minimize.

* **Linear Regression (OLS):** Tries to predict $y$ from $x$. It draws a line that minimizes the sum of squared **vertical distances** (residuals) between the points and the line.
* **PCA:** Doesn't predict anything; it treats $x$ and $y$ equally to find the axis of maximum variance. It minimizes the sum of squared **orthogonal (perpendicular) distances** from the points to the line.

Let's generate a small dataset so we can clearly draw and compare these distances side-by-side!

In [ ]:
# Generate a small dataset (fewer points make the lines easier to see)
x = np.random.uniform(-5, 5, 15)
y = x * 0.8 + np.random.normal(0, 1.5, 15)
data_2d = np.column_stack((x, y))

# Fit Linear Regression (OLS)
reg = LinearRegression().fit(x.reshape(-1, 1), y)
x_line = np.linspace(-6, 6, 100)
y_ols = reg.predict(x_line.reshape(-1, 1))
y_ols_predicted = reg.predict(x.reshape(-1, 1))

# Fit PCA
pca_vis = PCA(n_components=1).fit(data_2d)
mean_x, mean_y = pca_vis.mean_
v = pca_vis.components_[0] # The principal component vector
slope_pca = v[1] / v[0]
y_pca = slope_pca * (x_line - mean_x) + mean_y

# Calculate the orthogonal projections of our points onto the PCA line
# 1. Center the data
centered_data = data_2d - pca_vis.mean_
# 2. Project onto the principal component vector (dot product)
projections_centered = np.dot(centered_data, v).reshape(-1, 1) * v
# 3. Add the mean back to get coordinates in the original space
projected_points = projections_centered + pca_vis.mean_


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: OLS
ax1.scatter(x, y, color='black', zorder=3, s=50, label='Data Points')
ax1.plot(x_line, y_ols, color="blue", linewidth=2, label="OLS Line")
for i in range(len(x)):
    ax1.plot([x[i], x[i]], [y[i], y_ols_predicted[i]], color='red', linestyle='--', zorder=2, alpha=0.7)
ax1.set_title("Linear Regression (OLS)\nMinimizes Vertical Distance")
ax1.set_xlabel("Feature X")
ax1.set_ylabel("Feature Y")
ax1.legend()
ax1.axis('equal') # Equal aspect ratio ensures visual accuracy
ax1.grid(True, linestyle=':', alpha=0.6)

# Plot 2: PCA
ax2.scatter(x, y, color='black', zorder=3, s=50, label='Data Points')
ax2.plot(x_line, y_pca, color="green", linewidth=2, label="1st Principal Component")
for i in range(len(x)):
    ax2.plot([x[i], projected_points[i, 0]], [y[i], projected_points[i, 1]], color='red', linestyle='--', zorder=2, alpha=0.7)
ax2.set_title("PCA\nMinimizes Orthogonal (Perpendicular) Distance")
ax2.set_xlabel("Feature X")
ax2.set_ylabel("Feature Y")
ax2.legend()
ax2.axis('equal')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

## Task 1: Implement PCA on a Real Dataset (numpy only!)

You know the math, and you know the geometry. Now it is time to build it. 

Your task is to write a complete PCA pipeline from scratch using only `numpy`. We will use the **Wine Dataset**, which contains 13 different chemical features for 178 different wines. 13 dimensions is impossible to visualize, so your goal is to project it down to 2 dimensions.

Complete the four functions below. Once you are done, run the final testing cell. It will run your custom code and compare it against `scikit-learn` to see if your math holds up.

**Formulas to remember:**
1. **Standardize:** $X_{std} = \frac{X - \mu}{\sigma}$ (Subtract mean, divide by standard deviation)
2. **Covariance:** $\Sigma = \frac{1}{n-1} X_{std}^T X_{std}$
3. **Projection:** $X_{projected} = X_{std} W$ (where $W$ is the matrix of selected eigenvectors)

In [ ]:
# Load the Wine dataset
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print(f"Dataset loaded! Shape of X: {X_wine.shape}")

In [ ]:
def standardize_data(X: np.ndarray) -> np.ndarray:
    """Standardize the data to have zero mean and unit variance."""
    # TODO 1
    return X


def compute_covariance_matrix(X_std: np.ndarray) -> np.ndarray:
    """Compute the covariance matrix."""
    # TODO 2
    return X_std

def get_principal_components(cov_matrix: np.ndarray, n_components: int) -> np.ndarray:
    """Find and sort the eigenvectors, then return the top 'n_components'."""
    # TODO 3
    return cov_matrix


def project_data(X_std: np.ndarray, eigenvectors: np.ndarray) -> np.ndarray:
    """Project the standardized data onto the new eigenvectors."""
    # TODO 4
    return X_std


def pca_pipeline(X: np.ndarray, n_components: int) -> np.ndarray:
    """
    Run the full PCA pipeline from start to finish.
    Takes the raw data X and returns the projected data.
    """
    # TODO 5
    return X

In [ ]:
try:
    X_std_custom = standardize_data(X_wine)
    cov_mat_custom = compute_covariance_matrix(X_std_custom)
    eigenvectors_custom = get_principal_components(cov_mat_custom, n_components=2)
    X_projected_custom = project_data(X_std_custom, eigenvectors_custom)

    # Run Scikit-Learn's Pipeline
    # Note: sklearn's PCA automatically centers data, but doesn't scale by variance.
    # So we feed it our X_std_custom to ensure an apples-to-apples comparison.
    pca_sklearn = PCA(n_components=2)
    X_projected_sklearn = pca_sklearn.fit_transform(X_std_custom)

    print("Testing Standardization...")
    assert_almost_equal(np.mean(X_std_custom), 0.0, decimal=5, err_msg="Mean is not zero!")
    assert_almost_equal(np.std(X_std_custom), 1.0, decimal=5, err_msg="Standard deviation is not 1!")
    
    print("Testing Covariance Matrix Shape...")
    assert cov_mat_custom.shape == (13, 13), "Covariance matrix should be 13x13!"

    print("Testing Projection Math...")
    # We use np.abs() because eigenvectors can have flipped signs.
    assert_almost_equal(np.abs(X_projected_custom), np.abs(X_projected_sklearn), decimal=4, 
                        err_msg="Your projected data doesn't match scikit-learn!")

    print("\n🎉 ALL TESTS PASSED!")

except Exception as e:
    print(f"\n❌ TEST FAILED: {e}")
    print("Check your math and try again!")

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_projected_custom[:, 0], X_projected_custom[:, 1], c=y_wine, cmap='viridis', edgecolor='k', s=60)
plt.title("Your Custom PCA: Wine Dataset (2D Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label='Wine Class')
plt.show()

### Bonus Question

Why do we standardize if `scikit-learn` doesn't do that? Is it really necessary?

<details>
<summary><b>💡 Click for Hint</b></summary>

**Hint:** Think about comparing house prices (millions) to bedrooms (1-5). 

<details>
<summary><b>✅ Click for Answer</b></summary>

**Yes!** We standardize to prevent large numbers from completely dominating the variance. 

However, `scikit-learn` skips it by default because sometimes scale *is* the signal (like 0-255 pixel brightness in image compression, where standardizing would artificially amplify background noise).

</details>
</details>

In [ ]:
# Split the data into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42)

# Standardize the data (Fit on training, transform on both!)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

# Track our accuracies and calculate eigenvalues
accuracies = []
n_features = X_wine.shape[1]
component_range = range(1, n_features + 1)

# Loop to get accuracies
for n in component_range:
    pca = PCA(n_components=n)
    X_train_pca = pca.fit_transform(X_train_std)
    X_test_pca = pca.transform(X_test_std)
    
    svm = SVC(kernel='linear', random_state=42)
    svm.fit(X_train_pca, y_train)
    
    y_pred = svm.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

# Get the Eigenvalues (explained variance) from a full PCA run
full_pca = PCA().fit(X_train_std)
eigenvalues = full_pca.explained_variance_

In [ ]:
# Plotting Side-by-Side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Model Accuracy
ax1.plot(component_range, accuracies, marker='o', linewidth=3, markersize=8, color='#e74c3c')
for n, acc in zip(component_range, accuracies):
    ax1.text(n, acc + 0.015, f"{acc:.3f}", ha='center', va='bottom', 
             fontsize=10, fontweight='bold', color='#2c3e50')

ax1.axvline(x=2, color='gray', linestyle='--', alpha=0.7)
ax1.text(2.2, min(accuracies) + 0.05, 'Our 2D\nProjection', color='gray', fontsize=12)

ax1.set_title("Model Accuracy vs. Number of Components", fontsize=15)
ax1.set_xlabel("Number of Principal Components Kept", fontsize=12)
ax1.set_ylabel("Test Set Accuracy", fontsize=12)
ax1.set_xticks(component_range)
ax1.set_ylim(0.5, 1.10)
ax1.grid(True, linestyle=':', alpha=0.6)

# Plot 2: Eigenvalues (Scree Plot)
ax2.plot(component_range, eigenvalues, marker='s', linewidth=3, markersize=8, color='#2980b9')
for n, eig in zip(component_range, eigenvalues):
    ax2.text(n, eig + 0.15, f"{eig:.2f}", ha='center', va='bottom', 
             fontsize=10, fontweight='bold', color='#2c3e50')

# Adding the Kaiser Rule line
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7)
ax2.text(12.5, 1.15, 'Kaiser Rule\n(Eigenvalue = 1)', color='gray', ha='right', fontsize=11)

ax2.set_title("Scree Plot: Eigenvalues (Variance Explained)", fontsize=15)
ax2.set_xlabel("Principal Component Index", fontsize=12)
ax2.set_ylabel("Eigenvalue", fontsize=12)
ax2.set_xticks(component_range)
ax2.set_ylim(0, max(eigenvalues) + 0.8)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### The Takeaway
Look at the two plots side-by-side! 

On the right, we have a **Scree Plot**. It shows the eigenvalue (the amount of variance captured) by each principal component. Notice how the first 2 or 3 components hold almost all the massive eigenvalues, and then it violently flattens out? 

Now look at the accuracy plot on the left. The model's accuracy perfectly mirrors the eigenvalues! Once the eigenvalues drop below 1.0 (a threshold known as the **Kaiser Rule**), adding more principal components adds almost zero predictive power to our model. 

We threw away over 75% of our columns (going from 13 down to 3), but because PCA mathematically prioritized the axes of *maximum variance*, we kept almost 100% of the useful information needed to classify the wine.

## Task 2: Image Compression with PCA

Machine learning models aren't the only things that benefit from PCA. We can also use it for **data compression**. 

Think about a black-and-white image. It is literally just a 2D matrix of numbers, where each number represents the brightness of a pixel (from 0 for black to 255 for white). 

If a photo has a lot of empty sky, that is a lot of redundant, low-variance data. We can use PCA to find the "principal components" of the image, keep only the most important ones, and reconstruct the image using a fraction of the memory!

**Your Assignment:**
1. Complete the `compress_and_reconstruct` function below using `sklearn.decomposition.PCA`.
2. You will need to `fit_transform` the image to compress it.
3. Then, you will need to use `inverse_transform` to decompress it back into pixel coordinates so we can view it.

In [ ]:
china_color = load_sample_image("china.jpg")

# Convert it to Grayscale using a standard luminosity formula
# This turns our 3D color image into a simple 2D matrix
gray_china = np.dot(china_color[...,:3], [0.2989, 0.5870, 0.1140])

print(f"Original Image Shape: {gray_china.shape}")

In [ ]:
def compress_and_reconstruct(image: np.ndarray, n_components: int) -> np.ndarray:
    """
    Compresses a 2D image using PCA and then reconstructs it.
    
    Parameters:
    image (np.ndarray): The 2D grayscale image matrix.
    n_components (int): The number of principal components to keep.
    
    Returns:
    np.ndarray: The reconstructed image matrix.
    """
    # TODO 6

### The Visual Test

Let's test your function! We are going to compress the image using 10, 50, and 150 components and plot them side-by-side to see how much detail we retain. 

In [ ]:
components_to_test = [10, 50, 150]
plt.figure(figsize=(20, 6))

plt.subplot(1, 4, 1)
plt.imshow(gray_china, cmap='gray')
plt.title(f"Original Image\n(640 Components)")
plt.axis('off')

for i, n_comp in enumerate(components_to_test):
    reconstructed = compress_and_reconstruct(gray_china, n_comp)
    
    # Calculate the compression ratio (Memory saved)
    original_size = gray_china.shape[0] * gray_china.shape[1]
    compressed_size = (gray_china.shape[0] * n_comp) + (n_comp * gray_china.shape[1])
    saved = 100 * (1 - (compressed_size / original_size))
    
    # Plotting
    plt.subplot(1, 4, i+2)
    plt.imshow(reconstructed, cmap='gray')
    plt.title(f"{n_comp} Components\n({saved:.1f}% Memory Saved)")
    plt.axis('off')

plt.tight_layout()
plt.show()

<details>
<summary><b><font size="+1">Remember SVD?</font></b></summary>


Applying PCA on a single image is equivalent to using SVD on the mean-centered version of the image!
</details>

## Task 3: Fraud Detective: Rigorous Anomaly Detection with PCA

PCA has a hidden superpower: it is an incredible tool for finding anomalies, defects, and fraud.

**The Strategy (The Reconstruction Error Trick):**
1. We train a PCA model **only** on normal, safe data. The model learns the "rules" of normal variance.
2. We then take new data and compress/reconstruct it using our normal PCA model. 
3. Because the PCA model never learned the patterns of fraud, it does a terrible job reconstructing the fraudulent points. 
4. We calculate the distance between the original point and the reconstructed point (the **Reconstruction Error**). If the error is high, we flag it.

**The Rigorous Approach (Train / Val / Test):**
We cannot just guess the error threshold. We must use a proper ML split:
* **Train Set:** 100% pure normal data. Used *only* to train the PCA model.
* **Validation Set:** A mix of normal and fraud data. Used to programmatically find the exact threshold that separates the two.
* **Test Set:** A completely unseen mix of normal and fraud data. Used to prove our threshold actually works in the real world.

In [ ]:
# 1. Generate ALL Normal Transactions (Feature 1: Amount, Feature 2: Distance)
# The normal data is tightly clustered with a strong correlation (covariance of 0.8)
X_normal_all = np.random.multivariate_normal(mean=[0, 0], cov=[[1, 0.8], [0.8, 1]], size=200)

# 2. Generate ALL Fraudulent Transactions using a Uniform Distribution
# We scatter 10 fraud points randomly across a wide area (from -5 to 5 on both axes)
# Because they are completely random, they will ignore the "rules" of the normal PCA axis!
X_fraud_all = np.random.uniform(low=-5.0, high=5.0, size=(10, 2))

# TRAIN: 100 purely normal points
X_train = X_normal_all[:100]

# VALIDATION: 50 normal + 5 fraud (Labels: 0 for Normal, 1 for Fraud)
X_val = np.vstack((X_normal_all[100:150], X_fraud_all[:5]))
y_val = np.array([0] * 50 + [1] * 5) 

# TEST: 50 normal + 5 fraud
X_test = np.vstack((X_normal_all[150:], X_fraud_all[5:]))
y_test = np.array([0] * 50 + [1] * 5)

print(f"Train Set:      {X_train.shape[0]} samples (100% Normal)")
print(f"Validation Set: {X_val.shape[0]} samples (50 Normal, 5 Fraud)")
print(f"Test Set:       {X_test.shape[0]} samples (50 Normal, 5 Fraud)")

In [ ]:
def fit_normal_pca(X_train: np.ndarray, n_components: int) -> PCA:
    """Fit a PCA model ONLY on the training data."""
    # TODO 7
    return PCA()


def calculate_reconstruction_error(pca_model: PCA, X_new: np.ndarray) -> np.ndarray:
    """Compress and reconstruct the new data, then calculate the MSE per row."""
    # TODO 8
    return X_new


def find_best_threshold(val_errors: np.ndarray, y_val: np.ndarray) -> float:
    """
    Test 100 possible thresholds between the minimum and maximum validation errors.
    Return the threshold that yields the highest accuracy.
    """
    # TODO 9
    return 0.0


def detect_fraud_pipeline(X_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, 
                          X_test: np.ndarray, n_components: int = 1) -> tuple[np.ndarray, float]:
    """
    Runs the full PCA Anomaly Detection pipeline.
    
    Returns:
        test_predictions: A 1D numpy array of 0s and 1s representing your predictions for X_test.
        optimal_threshold: The threshold float value you found using the validation set.
    """
    # TODO 10
    return X_train, 0.0

In [ ]:
try:
    predictions, threshold = detect_fraud_pipeline(X_train, X_val, y_val, X_test, n_components=1)
    test_accuracy = accuracy_score(y_test, predictions)
    
    print("========================================")
    print("      🕵️ FRAUD PIPELINE RESULTS 🕵️      ")
    print("========================================")
    print(f"Optimal Threshold Found: {threshold:.3f}")
    print(f"Final Test Accuracy:     {test_accuracy * 100:.1f}%\n")

    assert len(predictions) == len(y_test), "Your predictions array must match the length of X_test!"
    assert test_accuracy > 0.90, "Accuracy is too low! Make sure you are using the validation threshold."
    assert set(predictions).issubset({0, 1}), "Predictions must be binary (0 or 1)!"
    
    print("🎉 ALL TESTS PASSED!\n")
except Exception as e:
    print(f"\n❌ PIPELINE FAILED: {e}")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Normal (0)", "Fraud (1)"]))

In [ ]:
plt.figure(figsize=(10, 6))

# Plot true Normal points in blue, true Fraud points in red
plt.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], c='#3498db', label="Actual Normal", s=80, alpha=0.6)
plt.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], c='#e74c3c', label="Actual Fraud", s=80, marker='x', linewidths=3)

# Highlight the points the model flagged as Fraud
fraud_flags = X_test[predictions == 1]
plt.scatter(fraud_flags[:, 0], fraud_flags[:, 1], facecolors='none', edgecolors='black', 
            s=200, linewidths=2, label="Flagged by Your PCA")

plt.title("Your Pipeline's Real-World Performance", fontsize=15)
plt.xlabel("Transaction Amount (Standardized)")
plt.ylabel("Distance from Home (Standardized)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()